# AdaBoost (Adaptive Boosting)

Course 3: Supervised Learning, Klasifikasi

AdaBoost adalah algoritma ensemble dari keluarga **boosting**. Seperti bagging dan random forest, AdaBoost menggabungkan banyak classifier, tapi caranya berbeda: classifier dilatih **berurutan**, dan tiap classifier baru lebih memperhatikan data yang salah diklasifikasikan oleh classifier sebelumnya (data itu diberi bobot lebih besar). Hasil akhirnya adalah gabungan berbobot dari semua weak classifier:

$$H(x) = \text{sign}\left(\sum_{t=1}^{T} \alpha_t h_t(x)\right)$$

dengan $h_t$ weak classifier ke-$t$ dan $\alpha_t$ bobotnya (makin akurat classifier-nya, makin besar bobotnya). Weak classifier default di scikit-learn adalah decision tree satu tingkat yang disebut **stump**.

## Dataset

Tiga dataset, semuanya dibaca langsung dari link:

- `ChurnData.csv`: pelanggan telekomunikasi, target `churn` (berhenti atau tidak), dipakai untuk menjelajahi AdaBoost
- `cell_samples.csv`: sampel sel tumor, target `Class` (2 jinak, 4 ganas), dipakai untuk grid search
- `drug200.csv`: pasien dan obat yang cocok untuknya, dipakai untuk latihan

Alamat `ChurnData.csv` dan `cell_samples.csv`: `https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/` (`drug200.csv` ada di folder yang sama).

## 1. Import

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
BASE = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/"


def akurasi(model, X_train, X_test, y_train, y_test):
    return {"train": accuracy_score(y_train, model.predict(X_train)),
            "test": accuracy_score(y_test, model.predict(X_test))}

## 2. Data churn

In [ ]:
churn = pd.read_csv(BASE + "ChurnData.csv")
churn = churn[["tenure", "age", "address", "income", "ed", "employ", "equip", "callcard", "wireless", "churn"]]
churn["churn"] = churn["churn"].astype(int)
print(churn.shape)
churn.head()

In [ ]:
fitur_churn = ["tenure", "age", "address", "income", "ed", "employ", "equip"]
X = churn[fitur_churn]
y = churn["churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)
print("train:", X_train.shape, "| test:", X_test.shape)

## 3. AdaBoost dengan 5 weak classifier

In [ ]:
ada5 = AdaBoostClassifier(n_estimators=5, random_state=0).fit(X_train, y_train)
print("akurasi AdaBoost (5 estimator):", {k: round(v, 3) for k, v in akurasi(ada5, X_train, X_test, y_train, y_test).items()})

Weak classifier-nya adalah decision tree satu tingkat (stump):

In [ ]:
print(ada5.estimator_)
print("jumlah weak classifier:", len(ada5.estimators_))
print("bobot tiap weak classifier (alpha):", ada5.estimator_weights_.round(3))

Tiap weak classifier sendirian tidak terlalu akurat. Kekuatannya muncul saat digabung:

In [ ]:
tabel = [{"weak classifier": i + 1, **{k: round(v, 3) for k, v in akurasi(h, X_train, X_test, y_train, y_test).items()}}
         for i, h in enumerate(ada5.estimators_)]
tabel.append({"weak classifier": "gabungan", **{k: round(v, 3) for k, v in akurasi(ada5, X_train, X_test, y_train, y_test).items()}})
pd.DataFrame(tabel).set_index("weak classifier")

## 4. Terlalu banyak weak classifier

In [ ]:
ada100 = AdaBoostClassifier(n_estimators=100, random_state=0).fit(X_train, y_train)
print("akurasi AdaBoost (100 estimator):", {k: round(v, 3) for k, v in akurasi(ada100, X_train, X_test, y_train, y_test).items()})

Menambah weak classifier bisa membuat model overfit. Untuk melihatnya, aku gambar akurasi train dan test terhadap jumlah estimator. Biar efisien, tiap model hanya dilatih sekali dengan 100 estimator, lalu `staged_score` memberi akurasi setelah tiap tahap boosting. Pengacakan train/test diulang 10 kali dan hasilnya dirata-ratakan.

In [ ]:
def kurva_boosting(X, y, learning_rates, ulang=10, n_max=100):
    train_acc = np.zeros((ulang, len(learning_rates), n_max))
    test_acc = np.zeros((ulang, len(learning_rates), n_max))
    for r in range(ulang):
        Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.3, random_state=r)
        for j, lr in enumerate(learning_rates):
            m = AdaBoostClassifier(n_estimators=n_max, learning_rate=lr, random_state=0).fit(Xa, ya)
            train_acc[r, j] = list(m.staged_score(Xa, ya))
            test_acc[r, j] = list(m.staged_score(Xb, yb))
    return train_acc.mean(axis=0), test_acc.mean(axis=0)


def plot_kurva(learning_rates, mean_train, mean_test):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharex=True)
    for j, lr in enumerate(learning_rates):
        axes[0].plot(range(1, mean_train.shape[1] + 1), mean_train[j], label=f"lr={lr}")
        axes[1].plot(range(1, mean_test.shape[1] + 1), mean_test[j], label=f"lr={lr}")
    axes[0].set(title="Akurasi train", xlabel="jumlah estimator", ylabel="akurasi")
    axes[1].set(title="Akurasi test", xlabel="jumlah estimator")
    axes[0].legend(); axes[1].legend()
    plt.tight_layout()
    plt.show()

tr1, te1 = kurva_boosting(X, y, [1])
plot_kurva([1], tr1, te1)

Kalau akurasi train terus naik sementara akurasi test mendatar atau turun, itu tanda overfitting: model makin cocok dengan data latih tapi tidak makin baik pada data baru.

## 5. Learning rate

`learning_rate` (default 1) mengecilkan kontribusi tiap weak classifier. Nilai yang lebih kecil membuat boosting lebih hati-hati dan bertindak sebagai regularisasi, tapi biasanya butuh lebih banyak estimator.

In [ ]:
ada_lr = AdaBoostClassifier(n_estimators=100, learning_rate=0.7, random_state=0).fit(X_train, y_train)
pd.DataFrame({
    "learning_rate=1.0": akurasi(ada100, X_train, X_test, y_train, y_test),
    "learning_rate=0.7": akurasi(ada_lr, X_train, X_test, y_train, y_test),
}).T.round(3)

In [ ]:
lrs = [0.2, 0.4, 0.6, 1]
tr_all, te_all = kurva_boosting(X, y, lrs)
plot_kurva(lrs, tr_all, te_all)

Dengan learning rate yang lebih kecil, kurva test biasanya lebih stabil, sedangkan akurasi train naik lebih lambat. Catatan: parameter `algorithm` (SAMME dan SAMME.R) yang ada di lab versi lama sudah tidak dipakai di scikit-learn terbaru, yang kini hanya memakai SAMME, jadi parameter itu tidak kusertakan.

## 6. Mengganti weak classifier

Weak classifier tidak harus stump. Tapi kalau classifier dasarnya terlalu kuat, boosting jadi cepat overfit dan hasilnya bisa lebih buruk. Sebagai percobaan, aku pakai SVM sebagai classifier dasar:

In [ ]:
svm_dasar = SVC(kernel="rbf", gamma=1).fit(X_train, y_train)
print("SVM sendirian     :", {k: round(v, 3) for k, v in akurasi(svm_dasar, X_train, X_test, y_train, y_test).items()})

ada_svm = AdaBoostClassifier(estimator=SVC(kernel="rbf", gamma=1), n_estimators=5, learning_rate=1, random_state=0)
ada_svm.fit(X_train, y_train)
print("AdaBoost + SVM    :", {k: round(v, 3) for k, v in akurasi(ada_svm, X_train, X_test, y_train, y_test).items()})

Bandingkan dua baris di atas. SVM tidak menghasilkan peluang kelas secara default, jadi AdaBoost memakai skema SAMME yang hanya butuh label prediksi. Selain itu, melatih banyak SVM jauh lebih lambat daripada melatih banyak stump.

## 7. Grid search pada data sel tumor

In [ ]:
sel = pd.read_csv(BASE + "cell_samples.csv")
sel = sel[pd.to_numeric(sel["BareNuc"], errors="coerce").notnull()].copy()
sel["BareNuc"] = sel["BareNuc"].astype(int)
print(sel.shape)

fitur_sel = ["Clump", "UnifSize", "UnifShape", "MargAdh", "SingEpiSize", "BareNuc", "BlandChrom", "NormNucl", "Mit"]
Xs, ys = sel[fitur_sel], sel["Class"]
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(Xs, ys, test_size=0.2, random_state=4)
print("train:", Xs_tr.shape, "| test:", Xs_te.shape)

In [ ]:
param_grid = {
    "learning_rate": [0.1 * (n + 1) for n in range(10)],
    "n_estimators": [2 * n + 1 for n in range(10)],
}
cari = GridSearchCV(AdaBoostClassifier(random_state=0), param_grid, scoring="accuracy", cv=5, n_jobs=-1)
cari.fit(Xs_tr, ys_tr)

print("skor cross validation terbaik:", round(cari.best_score_, 4))
print("parameter terbaik            :", cari.best_params_)
print("akurasi:", {k: round(v, 4) for k, v in akurasi(cari.best_estimator_, Xs_tr, Xs_te, ys_tr, ys_te).items()})

## 8. Latihan: data obat (drug200)

Datanya berisi usia, jenis kelamin, tekanan darah (BP), kolesterol, dan rasio Na terhadap K pasien. Targetnya jenis obat yang cocok (lima kelas). Kolom kategorikal diubah jadi angka dengan `OrdinalEncoder`, dan urutan kategorinya ditentukan agar sesuai tingkatannya (LOW < NORMAL < HIGH).

In [ ]:
drug = pd.read_csv(BASE + "drug200.csv")
print(drug.shape)

enc = OrdinalEncoder(categories=[["F", "M"], ["LOW", "NORMAL", "HIGH"], ["NORMAL", "HIGH"]])
Xd = drug[["Age", "Sex", "BP", "Cholesterol", "Na_to_K"]].copy()
Xd[["Sex", "BP", "Cholesterol"]] = enc.fit_transform(Xd[["Sex", "BP", "Cholesterol"]])
yd = drug["Drug"]

Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.2, random_state=4)
print("train:", Xd_tr.shape, "| test:", Xd_te.shape)

In [ ]:
cari_drug = GridSearchCV(AdaBoostClassifier(random_state=0), param_grid, scoring="accuracy", cv=3, n_jobs=-1)
cari_drug.fit(Xd_tr, yd_tr)

print("skor cross validation terbaik:", round(cari_drug.best_score_, 4))
print("parameter terbaik            :", cari_drug.best_params_)
print("akurasi:", {k: round(v, 4) for k, v in akurasi(cari_drug.best_estimator_, Xd_tr, Xd_te, yd_tr, yd_te).items()})

## Ringkasan

- AdaBoost melatih weak classifier berurutan, dan tiap classifier baru fokus pada data yang sebelumnya salah diklasifikasikan.
- Terlalu banyak estimator dapat menyebabkan overfitting. `learning_rate` yang lebih kecil membantu mengendalikannya.
- Classifier dasar yang terlalu kuat (seperti SVM) tidak cocok untuk boosting.
- `n_estimators` dan `learning_rate` dipilih dengan `GridSearchCV` di data train.